# OpenAI Function Calling In LangChain

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

Pydantic is the most widely used data validation library for Python.

The schema that Pydantic validates against is generally defined by Python type hints. Pydantic data classes are a blend of Python's data classes with the validation power of Pydantic - they offer a concise way to define data structures while ensuring that the data adheres to specified types and constraints.


In [2]:
from pydantic import BaseModel

In [3]:
class User:
    def __init__(self, name: str, age: int, email: str):
        self.name = name
        self.age = age
        self.email = email

In [4]:
example = User("Akhilesh", 20, "akhileshmasali488@gmail.com")

In [6]:
example.name

'Akhilesh'

In [7]:
example2 = User("Mallikarjun", "string", "xyz@gmail.com")
example2.age

'string'

We said `age` is meant to be an `int`, but it still accepted a string. A plain Python class like this has no real validation - Pydantic is what lets us actually enforce that constraint.

## What is `BaseModel`?

`BaseModel` is the base class every Pydantic model inherits from. Just by inheriting from it and declaring your fields with type hints (like we did with `pUser` above), you get:
* automatic data validation - if you pass in the wrong type, Pydantic raises an error instead of silently accepting it (unlike the plain `User` class from before)
* automatic type coercion where sensible (e.g. a numeric string can get converted to an `int`)
* a bunch of built-in functionality for free - things like `.dict()`, `.json()`, and schema generation

Basically, it's what turns a plain data class into something that actively enforces the structure and types you defined for it.


In [8]:
class pUser(BaseModel):
    name: str
    age: int
    email: str

In [10]:
ex = pUser(name = "Patrick Jane", age = 34, email = "Jane@gmail.com")
ex.age

34

In [11]:
#if we try to pass in values of different data types we should get an error
ex2 = pUser(name = 32, age = "Jane", email = "xyz")
ex2.age

ValidationError: 1 validation error for pUser
age
  value is not a valid integer (type=type_error.integer)

You can also play with nested models, like this:

In [ ]:
class Class(BaseModel):
    students: list[pUser]

In [15]:
obj = Class(
    students = [pUser(name = "A", age = 20, email = "a@gmail.com")]
)
obj

Class(students=[pUser(name='A', age=20, email='a@gmail.com')])

## Pydantic to OpenAI function defination

Instead of writing out a long function description by hand for the LLM, you can use Pydantic to define it - it makes the work a lot easier.

In [19]:
from pydantic import Field #Field is used to set the description for the arguments
from langchain.utils.openai_functions import convert_pydantic_to_openai_function

In [18]:
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str = Field(description="airport code to get weather for")

In [20]:
func = convert_pydantic_to_openai_function(WeatherSearch)

In [21]:
func

{'name': 'WeatherSearch',
 'description': 'Call this with an airport code to get the weather at that airport',
 'parameters': {'title': 'WeatherSearch',
  'description': 'Call this with an airport code to get the weather at that airport',
  'type': 'object',
  'properties': {'airport_code': {'title': 'Airport Code',
    'description': 'airport code to get weather for',
    'type': 'string'}},
  'required': ['airport_code']}}

### What `convert_pydantic_to_openai_function` actually does

It takes a Pydantic class (like `WeatherSearch`) and automatically converts it into the JSON schema format that OpenAI's function-calling API expects - the same `{"name": ..., "description": ..., "parameters": {...}}` structure we wrote out by hand in the previous notebook.

Specifically, it maps:
* the class name (`WeatherSearch`) → the function's `"name"`
* the class docstring → the function's `"description"`
* each field's type hint (`str`, `int`, etc.) → that parameter's JSON schema `"type"`
* each field's `Field(description=...)` → that parameter's `"description"`
* any field without a default value → gets added to `"required"` automatically

So instead of writing out that whole nested dictionary by hand (and keeping it in sync with your function's actual arguments), you just define a Pydantic class once and let LangChain generate the schema for you.


The `description` inside `Field()` is optional for each argument, but the docstring for the class itself must be specified - otherwise you'll get an error, since that's what becomes the function's required `"description"`.

In [23]:
from langchain.chat_models import ChatOpenAI

In [24]:
model = ChatOpenAI()

In [26]:
model.invoke("what is the weather is SF?", functions = [func])

AIMessage(content='', additional_kwargs={'function_call': {'name': 'WeatherSearch', 'arguments': '{"airport_code":"SFO"}'}})

Another way to do the same thing:

In [27]:
model_with_func = model.bind(functions = [func])

In [28]:
model_with_func.invoke("what is the weather is SF?")

AIMessage(content='', additional_kwargs={'function_call': {'name': 'WeatherSearch', 'arguments': '{"airport_code":"SFO"}'}})

## Using chain

In [29]:
from langchain.prompts import ChatPromptTemplate

In [39]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful assistant"),
        ("user", "{input}")
    ]
)

In [40]:
chain = prompt | model_with_func

In [42]:
chain.invoke({"input":"Hi"})

AIMessage(content='Hello! How can I assist you today?')

In [44]:
chain.invoke({"input": "What is the weather is SF?"})

AIMessage(content='', additional_kwargs={'function_call': {'name': 'WeatherSearch', 'arguments': '{"airport_code":"SFO"}'}})

## Using multiple functions

In [45]:
class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""
    artist_name: str = Field(description="name of artist to look up")
    n: int = Field(description="number of results")

In [46]:
functions = [
    convert_pydantic_to_openai_function(WeatherSearch),
    convert_pydantic_to_openai_function(ArtistSearch)
]

In [49]:
model_with_funcs = model.bind(functions = functions)

model_with_funcs.invoke("what is the weather in SF?")

AIMessage(content='', additional_kwargs={'function_call': {'name': 'WeatherSearch', 'arguments': '{"airport_code":"SFO"}'}})

In [50]:
model_with_funcs.invoke("what are the top three songs of Arijit Singh?")

AIMessage(content='', additional_kwargs={'function_call': {'name': 'ArtistSearch', 'arguments': '{"artist_name":"Arijit Singh","n":3}'}})

In [ ]:
model_with_funcs.invoke("Hi")